# 02 · Autómatas celulares: de las reglas elementales a la criticalidad

Taller «Entre el orden y el caos». Corre de principio a fin en **Google Colab** o en tu computadora.
Todo el azar tiene semilla fija.

1. **Autómatas elementales**: Reglas 30, 90 y 110, y un barrido de las 256 reglas con dos medidas de
   complejidad aparente (compresión y entropía).
2. **Pila de arena** (Bak, Tang y Wiesenfeld 1987) e **incendio forestal** (Drossel y Schwabl 1992):
   autómatas celulares que se organizan solos hacia un estado crítico.
3. **¿Ley de potencias?** Ajuste por máxima verosimilitud (Clauset, Shalizi y Newman 2009) y la
   prueba que de verdad importa en un sistema finito: **variar el tamaño**.

Los scripts equivalentes están en `simulaciones/`; el método, en `skills/ajuste-ley-de-potencias/SKILL.md`.

In [ ]:
# Configuración: ¿Colab o local? ¿Dónde se guardan figuras y números?
import sys, json, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Cambia esta ruta por la carpeta del kit en tu Drive (Mi unidad/kit-taller).
    RAIZ = Path("/content/drive/MyDrive/kit-taller")
else:
    # Local: el notebook vive en notebooks/, la raíz del repositorio es la carpeta de arriba.
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FIG = RAIZ / "figuras"
RES = RAIZ / "resultados"
FIG.mkdir(parents=True, exist_ok=True)
RES.mkdir(parents=True, exist_ok=True)
print("Colab:", EN_COLAB, "| figuras en", FIG, "| números en", RES)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TINTA, TINTA2, AZUL, NARANJA, AQUA = "#0b0b0b", "#52514e", "#2a78d6", "#eb6834", "#1baf7a"
SECUENCIAL = ["#9cc3ee", "#5f9de4", "#2a78d6", "#1b4f8f"]   # un solo tono, para series ordenadas
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e1", "font.size": 9, "legend.frameon": False})

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(FIG / nombre, dpi=200)
    print("figura ->", FIG / nombre)

RESULTADOS = {}
def guardar_resultados(nombre):
    (RES / nombre).write_text(json.dumps(RESULTADOS, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
    print("números ->", RES / nombre)
import zlib, math, time
from matplotlib.colors import ListedColormap

# numba compila los bucles de la pila de arena y del incendio. En Colab ya viene instalado.
# Si falta, el código corre igual pero mucho más lento (y usamos tamaños más pequeños).
try:
    from numba import njit
    HAY_NUMBA = True
except ImportError:
    HAY_NUMBA = False
    def njit(*a, **k):
        return a[0] if a and callable(a[0]) else (lambda f: f)
print("numba:", HAY_NUMBA)

# powerlaw (ajuste de leyes de potencias) no viene en Colab: se instala si falta.
try:
    import powerlaw
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "powerlaw"], check=True)
    import powerlaw
import warnings
warnings.filterwarnings("ignore")

## 1. Autómatas celulares elementales

Cada celda mira a su vecina izquierda, a sí misma y a su vecina derecha (8 combinaciones); la regla, un número de 0 a 255, dice en binario qué sale en cada caso.

In [ ]:
SEMILLA_AC = 2026

def paso(estado, regla):
    vecindad = (np.roll(estado, 1) << 2) | (estado << 1) | np.roll(estado, -1)
    return (regla >> vecindad) & 1

def simular(regla, ancho, pasos, inicio="centro", rng=None, posicion=None):
    if inicio == "centro":
        e = np.zeros(ancho, dtype=np.int64)
        e[ancho // 2 if posicion is None else posicion] = 1
    else:
        e = rng.integers(0, 2, ancho).astype(np.int64)
    d = np.empty((pasos, ancho), dtype=np.int64)
    for t in range(pasos):
        d[t] = e
        e = paso(e, regla)
    return d

# Control de corrección: la Regla 90 desde una celda es el triángulo de Pascal módulo 2.
n = 64
d = simular(90, 2 * n + 1, n)
pascal_ok = all(d[t, n - t + 2 * j] == math.comb(t, j) % 2 for t in range(n) for j in range(t + 1))
print("Regla 90 = Pascal mod 2:", pascal_ok)

fig, ejes = plt.subplots(1, 3, figsize=(8, 2.2))
for ax, r, pos in zip(ejes, [30, 90, 110], [None, None, 158]):   # la 110 crece hacia la izquierda
    ax.imshow(simular(r, 161, 80, posicion=pos), cmap=ListedColormap(["white", TINTA]), interpolation="nearest")
    ax.set_title(f"Regla {r}")
    ax.axis("off")
guardar(fig, "nb02_reglas.png")

### Barrido de las 256 reglas

Para cada regla, desde una condición inicial al azar, medimos la **razón de compresión** zlib (≈ 0 muy
regular, ≈ 1 incompresible) y la **entropía de bloques** de 6 celdas, normalizada.

In [ ]:
ANCHO, PASOS, TRANSITORIO, K = 256, 256, 64, 6

def compresion(d):
    crudo = np.packbits(d.astype(np.uint8)).tobytes()
    return len(zlib.compress(crudo, 9)) / len(crudo)

def entropia_bloques(d, k=K):
    ventanas = sum(np.roll(d, -i, axis=1) << (k - 1 - i) for i in range(k)).ravel()
    p = np.bincount(ventanas, minlength=2 ** k) / ventanas.size
    p = p[p > 0]
    return max(0.0, float(-(p * np.log2(p)).sum() / k))

def barrido(ancho, semilla):
    rng = np.random.default_rng(semilla)
    comp, ent = np.empty(256), np.empty(256)
    for r in range(256):
        d = simular(r, ancho, PASOS, "azar", rng)[TRANSITORIO:]
        comp[r], ent[r] = compresion(d), entropia_bloques(d)
    return comp, ent

t0 = time.time()
comp, ent = barrido(ANCHO, SEMILLA_AC)
print(f"barrido en {time.time() - t0:.0f} s; reglas con compresión < 0.1: {(comp < 0.1).sum()}")

REFERENCIA = {0: "I", 4: "II", 108: "II", 30: "III", 90: "III", 110: "IV"}
colores = {"I": "#8a8984", "II": AQUA, "III": NARANJA, "IV": AZUL}
fig, ax = plt.subplots(figsize=(5.6, 4))
ax.scatter(comp, ent, s=14, c=TINTA2, alpha=0.35, edgecolor="none", label="256 reglas")
for r, c in REFERENCIA.items():
    ax.scatter(comp[r], ent[r], s=60, c=colores[c], edgecolor="white", zorder=3)
    ax.annotate("0 y 255" if r == 0 else str(r), (comp[r], ent[r]), xytext=(5, 4), textcoords="offset points", fontsize=7)
for c, col in colores.items():
    ax.scatter([], [], c=col, s=40, label=f"clase {c} (referencia)")
ax.set(xlabel="razón de compresión (zlib)", ylabel="entropía de bloques, normalizada")
ax.legend(fontsize=7, loc="lower right")
guardar(fig, "nb02_barrido.png")
RESULTADOS["elementales"] = {"pascal_mod2_verificado": bool(pascal_ok), "semilla": SEMILLA_AC, "ancho": ANCHO,
                             "reglas_con_compresion_menor_0_1": int((comp < 0.1).sum()),
                             "referencia": {str(r): {"compresion": round(float(comp[r]), 3),
                                                     "entropia": round(float(ent[r]), 3)} for r in REFERENCIA}}

**Tu turno.** Repite el barrido con `ancho = 257` en lugar de 256 y compara la Regla 90. ¿Una
medida de complejidad debería depender de que el ancho sea potencia de 2? Eso es una **prueba de
sensibilidad**, y su resultado va al paper aunque «estorbe».

## 2. Pila de arena: avalanchas de todos los tamaños

Cada grano cae en una celda al azar. Una celda con 4 granos se derrumba y reparte uno a cada vecina,
que puede derrumbarse a su vez: una **avalancha**. En el borde, la arena se cae.

In [ ]:
@njit(cache=True)
def pila_de_arena(L, granos, semilla):
    np.random.seed(semilla)
    z = np.zeros((L, L), np.int64)
    tam = np.zeros(granos, np.int64)
    pila = np.empty(L * L * 8, np.int64)
    for g in range(granos):
        i = np.random.randint(L)
        j = np.random.randint(L)
        z[i, j] += 1
        n = 0
        tope = 0
        if z[i, j] >= 4:
            pila[0] = i * L + j
            tope = 1
        while tope > 0:
            tope -= 1
            k = pila[tope]
            a = k // L
            b = k % L
            if z[a, b] < 4:
                continue
            z[a, b] -= 4
            n += 1
            if z[a, b] >= 4:
                pila[tope] = k
                tope += 1
            for da, db in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                x = a + da
                y = b + db
                if 0 <= x < L and 0 <= y < L:
                    z[x, y] += 1
                    if z[x, y] == 4:
                        pila[tope] = x * L + y
                        tope += 1
        tam[g] = n
    return tam, z

SEMILLA_ARENA = 3
TAMANOS = [16, 32, 64] if HAY_NUMBA else [8, 16, 32]

def pdf_log(x, nbins=35):
    x = np.asarray(x, float)
    bordes = np.logspace(0, np.log10(x.max()) + 0.01, nbins)
    h, _ = np.histogram(x, bordes)
    centro = np.sqrt(bordes[1:] * bordes[:-1])
    m = h > 0
    return centro[m], (h / (np.diff(bordes) * x.size))[m]

avalanchas = {}
fig, ax = plt.subplots(figsize=(6, 4))
for L, color in zip(TAMANOS, SECUENCIAL[1:]):
    t0 = time.time()
    granos = 60 * L * L + 100_000
    tam, z = pila_de_arena(L, granos, SEMILLA_ARENA)
    aval = tam[granos // 3:]
    avalanchas[L] = aval = aval[aval > 0]
    x, y = pdf_log(aval)
    ax.loglog(x, y, "o-", ms=3, lw=1, color=color, label=f"L = {L}")
    print(f"L = {L}: {aval.size} avalanchas, máxima {aval.max()}  ({time.time() - t0:.1f} s)")
ax.set(xlabel="tamaño de avalancha s", ylabel="P(s)", title="El corte crece con el tamaño del sistema")
ax.legend()
guardar(fig, "nb02_arena_tamano_finito.png")
RESULTADOS["pila_de_arena"] = {"semilla": SEMILLA_ARENA,
                               "maxima_por_L": {str(L): int(a.max()) for L, a in avalanchas.items()}}

## 3. ¿Es una ley de potencias? El método y su trampa

Una recta en log-log **no basta**. El método de Clauset, Shalizi y Newman (2009):
1. elegir $x_{min}$ minimizando la distancia de Kolmogorov-Smirnov;
2. estimar $\alpha$ por máxima verosimilitud;
3. comparar con alternativas (log-normal, exponencial, ley de potencias **con corte**) por razón de
   verosimilitud: $R > 0$ favorece a la primera, y solo cuenta si $p < 0.05$.

En un sistema finito, el paso 1 puede elegir $x_{min}$ justo donde el sistema se corta. Por eso
comparamos también con $x_{min}$ fijo y bajo, contra una ley **con corte**.

Para que tarde segundos: la búsqueda de $x_{min}$ prueba 40 candidatos en escala logarítmica, y
ajustamos una submuestra de 20 000 avalanchas con semilla fija (se declara en los resultados).

In [ ]:
def comparar(f, a, b):
    R, p = f.distribution_compare(a, b, normalized_ratio=True)
    return {"R": round(float(R), 2), "p": float(f"{float(p):.3g}")}   # p con 3 cifras significativas

L = TAMANOS[-1]
datos = np.random.default_rng(SEMILLA_ARENA).choice(avalanchas[L], min(20_000, avalanchas[L].size), replace=False)

# Paso 1: x_min automático (rejilla logarítmica de candidatos, distancia KS)
valores = np.unique(datos)[:-10]
objetivo = np.logspace(0, np.log10(valores[-1]), 40)
candidatos = np.unique(valores[np.clip(np.searchsorted(valores, objetivo), 0, valores.size - 1)])
auto = min((powerlaw.Fit(datos, discrete=True, xmin=float(c), verbose=False) for c in candidatos),
           key=lambda f: f.power_law.D)
print(f"Automático: x_min = {auto.xmin:.0f} (quedan {np.sum(datos >= auto.xmin)} datos en la cola), α = {auto.power_law.alpha:.2f}")
print("   ley de potencias vs log-normal:", comparar(auto, "power_law", "lognormal"))

# Paso 2: x_min fijo y bajo; ley pura vs. con corte vs. log-normal
fijo = powerlaw.Fit(datos, discrete=True, xmin=10, verbose=False)
c_pura, c_logn = comparar(fijo, "truncated_power_law", "power_law"), comparar(fijo, "truncated_power_law", "lognormal")
print(f"Con x_min = 10: α con corte = {fijo.truncated_power_law.alpha:.2f}, escala del corte 1/λ = {1 / fijo.truncated_power_law.parameter2:.0f}")
print("   con corte vs pura:", c_pura, "| con corte vs log-normal:", c_logn)

fig, ax = plt.subplots(figsize=(6, 4))
fijo.plot_ccdf(ax=ax, color=AZUL, lw=1.6, label="datos (x ≥ 10)")
fijo.power_law.plot_ccdf(ax=ax, color=TINTA2, ls="--", lw=1, label=f"ley pura, α = {fijo.power_law.alpha:.2f}")
fijo.truncated_power_law.plot_ccdf(ax=ax, color=NARANJA, lw=1.2, label="ley con corte")
fijo.lognormal.plot_ccdf(ax=ax, color=AQUA, ls=":", lw=1.4, label="log-normal")
ax.set(xlabel="tamaño de avalancha s", ylabel="P(S ≥ s)", title=f"Pila de arena, L = {L}")
ax.legend(fontsize=7)
guardar(fig, "nb02_arena_ajuste.png")
RESULTADOS["ajuste_arena"] = {"L": L, "eventos_ajustados": int(datos.size),
    "automatico": {"xmin": float(auto.xmin), "alfa": round(float(auto.power_law.alpha), 3),
                   "ley_vs_lognormal": comparar(auto, "power_law", "lognormal")},
    "xmin_10": {"alfa_con_corte": round(float(fijo.truncated_power_law.alpha), 3),
                "corte_vs_pura": c_pura, "corte_vs_lognormal": c_logn}}

## 4. Incendio forestal: el corte lo fija θ

Se planta un árbol por paso; cada θ árboles cae un rayo y, si da en un árbol, se quema todo el bosque
conectado. Aquí el corte de la distribución lo fija θ (no L, mientras L sea suficientemente grande).

In [ ]:
@njit(cache=True)
def incendio(L, plantados, theta, semilla):
    np.random.seed(semilla)
    bosque = np.zeros(L * L, np.uint8)
    fuegos = np.empty(plantados // theta + 1, np.int64)
    nf = 0
    cola = np.empty(L * L, np.int64)
    for t in range(plantados):
        k = np.random.randint(L * L)
        bosque[k] = 1
        if (t + 1) % theta == 0:
            k = np.random.randint(L * L)
            if bosque[k] == 1:
                bosque[k] = 0
                cola[0] = k
                ini = 0
                fin = 1
                while ini < fin:
                    c = cola[ini]
                    ini += 1
                    a = c // L
                    b = c % L
                    for da, db in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        x = a + da
                        y = b + db
                        if 0 <= x < L and 0 <= y < L and bosque[x * L + y] == 1:
                            bosque[x * L + y] = 0
                            cola[fin] = x * L + y
                            fin += 1
                fuegos[nf] = fin
                nf += 1
    return fuegos[:nf], bosque.reshape(L, L)

SEMILLA_BOSQUE = 5
L_BOSQUE = 128 if HAY_NUMBA else 48
THETAS = [30, 100, 300] if HAY_NUMBA else [10, 30]
maximos = {}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.8), gridspec_kw={"width_ratios": [1.4, 1]})
for theta, color in zip(THETAS, SECUENCIAL[1:]):
    t0 = time.time()
    fuegos, bosque = incendio(L_BOSQUE, 20_000 * theta + 500_000, theta, SEMILLA_BOSQUE)
    fuegos = fuegos[len(fuegos) // 5:]
    x, y = pdf_log(fuegos)
    ax1.loglog(x, y, "o-", ms=3, lw=1, color=color, label=f"θ = {theta}")
    maximos[str(theta)] = float(np.percentile(fuegos, 99.9))
    print(f"θ = {theta}: {fuegos.size} incendios, percentil 99.9 = {maximos[str(theta)]:.0f}  ({time.time() - t0:.1f} s)")
ax1.set(xlabel="árboles quemados s", ylabel="P(s)", title=f"Incendio forestal, L = {L_BOSQUE}")
ax1.legend()
ax2.imshow(bosque, cmap="Greens", vmin=0, vmax=1.3, interpolation="nearest")
ax2.set_title(f"bosque final, θ = {THETAS[-1]}")
ax2.axis("off")
guardar(fig, "nb02_incendio.png")
RESULTADOS["incendio"] = {"semilla": SEMILLA_BOSQUE, "L": L_BOSQUE, "percentil_99_9_por_theta": maximos}
guardar_resultados("nb02_automatas.json")

## Tu turno: ¿qué puedes afirmar?

Con la tabla de `skills/ajuste-ley-de-potencias/SKILL.md`, decide qué frase puedes escribir sobre la pila
de arena y cuál **no**:

- ¿ganó la ley con corte contra la pura y contra la log-normal (con $p < 0.05$)?
- ¿el corte crece con L? ¿con cuántos tamaños lo comprobaste?
- ¿qué eligió el ajuste automático y por qué eso sería engañoso?

Escribe también **lo que no funcionó**. Después, en Terminus: «@simulador corre
`simulaciones/pila_de_arena.py` y lleva la figura de tamaño finito al paper», y «@redactor escribe los
resultados con esos números».